load df:

In [2]:
import pandas as pd
from py_scripts.music_utils import draw_piano_roll, create_piano_roll

#load data from TSV
df = pd.read_csv('tsv/test_bach_wtk2_5.tsv', delimiter='\t')

# Extract measure offsets for measure lines
# For measure 1, use 0.0; for other measures, use the minimum onset time
measure_min_onsets = df.groupby('Measure')['Global Onset'].min()
measure_offsets = []

for measure_num in sorted(df['Measure'].unique()):
    if measure_num == 1:
        # First measure should start at 0.0
        measure_offsets.append(0.0)
    else:
        # Other measures use their minimum onset time
        measure_offsets.append(measure_min_onsets[measure_num])

display(df)

# Now both backends support pixel dimensions!
backend_choice = "bokeh"  # Change to "plt" or "bokeh"

fig = draw_piano_roll(
    df,
    measure_offsets=measure_offsets,
    backend=backend_choice,
    show=True,
    show_measure_lines=True, # show measure lines
    plot_width=1200,        # Width in pixels (works for both backends!)
    plot_height=800,        # Height in pixels (works for both backends!)
    dpi=100,                # DPI for matplotlib conversion (ignored for bokeh)
    zoom_drag_dim="both",   # Box zoom drag tool, select "width" or "height" or "both" (bokeh only)
    zoom_wheel_dim="width",  # Wheel zoom tool, select "width" or "height" or "both" (bokeh only)
)

,Measure,Local Onset,Global Onset,Duration,Pitch,MIDI,Voice
0,1,0.5,0.5,0.5,D4,62,spine_0
1,1,1.0,1.0,0.5,D4,62,spine_0
2,1,1.5,1.5,0.5,D4,62,spine_0
3,1,2.0,2.0,1.0,G3,55,spine_0
4,1,3.0,3.0,1.0,B3,59,spine_0
...,...,...,...,...,...,...,...
964,50,1.5,197.5,0.5,G3,55,spine_0 / Voice 2543746544992
965,50,2.0,198.0,2.0,F#3,54,spine_0 / Voice 2543746544992
966,50,2.0,198.0,2.0,D4,62,spine_0
967,50,2.0,198.0,2.0,A3,57,spine_0


In [3]:
# Binary conversion and visualization (refactored to py_scripts)
import numpy as np
import pandas as pd
from py_scripts.music_utils import create_binary_matrix, plot_binary_matrix

# Configuration
BINARY_RESOLUTION_METHOD = 'manual'   # 'auto' | 'manual' | 'standard'
BINARY_MANUAL_RES = 0.5            # e.g., 0.5 when using 'manual'
Y_AXIS_MODE = 'full'              # 'full' | 'minmax' | 'chroma'
Y_AXIS_MIN = 60                   # optional int when Y_AXIS_MODE=='minmax'
Y_AXIS_MAX = 71                   # optional int when Y_AXIS_MODE=='minmax'

PLOTTING_BACKEND = 'bokeh'  # 'plt' | 'bokeh' | 'none'

# Build binary matrix
binary_matrix_df, meta = create_binary_matrix(
    df,
    resolution_method=BINARY_RESOLUTION_METHOD,
    manual_resolution=BINARY_MANUAL_RES,
    y_mode=Y_AXIS_MODE,
    midi_low=Y_AXIS_MIN,
    midi_high=Y_AXIS_MAX,
    row_order="high_to_low"
)

print(f"Binary matrix shape: {binary_matrix_df.shape}")
print(f"Resolution: {meta['resolution']}, columns: {meta['num_cols']}, y_mode: {meta['y_mode']}")

# Plot using the same backend as parsing cell (if defined), else default to 'plt'
backend_to_use = PLOTTING_BACKEND

# If measure offsets were computed earlier, you can pass them here.
# In this notebook, measure offsets are inside `results` items if needed; we default to None
plot_binary_matrix(
    binary_matrix_df,
    meta,
    backend=backend_to_use,
    measure_offsets=None,
    show_measure_lines=True,
    show=True,
    plot_width=1200,       # Width in pixels (works for both backends!)
    plot_height=800,       # Height in pixels (works for both backends!)
    dpi=100,
    zoom_drag_dim="both",   # Box zoom drag tool, select "width" or "height" or "both" (bokeh only)
    zoom_wheel_dim="width",  # Wheel zoom tool, select "width" or "height" or "both" (bokeh only)               # DPI for matplotlib conversion (ignored for bokeh)
)

Binary matrix shape: (128, 404)
Resolution: 0.5, columns: 404, y_mode: full


figure(id='p1102', ...)

In [4]:
print(binary_matrix_df)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [1]:
from py_scripts.binary_matrix_designer import binary_matrix_designer

# Start with an empty 12x12 grid with pitch names enabled
ui = binary_matrix_designer(
    rows=12,
    cols=12,
    # prototype=None,        # default
    # prototype_meta=None,   # default
    flip_vertical=True,
    display_ui=True         # or False + display(ui)
)

# The pitch names checkbox is now integrated into the "Matrix Setup" section!
# - Check "Show pitch names" to display C, C#, D, D#, E, F, F#, G, G#, A, A#, B
# - The bottom row shows "C", and it cycles through the chromatic scale
# - Uncheck to hide the pitch names
# - Located in the Matrix Setup section below the main controls


In [6]:
print(binary_matrix_ui)

NameError: name 'binary_matrix_ui' is not defined

In [ ]:
# Pattern search metrics: convolution and cross-correlation between binary_matrix_df and binary_matrix_ui
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.palettes import Viridis256
from tqdm.auto import tqdm

STRIDE_Y = 1  # step along pitch axis (rows); increase to prefer octave/tonality jumps
STRIDE_X = 1  # step along time axis (columns)
TOP_N_MATCHES = 20  # number of highest-scoring positions to report per metric
PLOT_THRESHOLDS = {
    'normalized_overlap': 0.7,  # e.g., 0.6 to keep only strong overlaps
    'cross_covariance': None,
    'normalized_cross_correlation': None,
}
MPL_CMAP = 'viridis'

METRICS_TO_RUN = ['normalized_overlap'] # 'normalized_overlap', 'cross_covariance', 'normalized_cross_correlation'
KERNEL_SCALE_FACTORS = [0.5, 0.75, 1.0, 1.5, 2.0]  # e.g., [0.5, 0.75, 1.0, 1.5, 2.0]
KERNEL_SCALE_AXES = ['x']  # options: 'x', 'y', 'both'
PLOT_SCALED_KERNELS = False

backend_to_use = globals().get('PLOTTING_BACKEND', 'plt')
try:
    output_notebook(hide_banner=True)
except TypeError:
    output_notebook()

AVAILABLE_METRICS = {
    'normalized_overlap': {'label': 'Normalised overlap'},
    'cross_covariance': {'label': 'Cross-covariance'},
    'normalized_cross_correlation': {'label': 'Normalised cross-correlation'},
}

if 'binary_matrix_df' not in globals() or 'binary_matrix_ui' not in globals():
    raise NameError('Run the previous cells to define binary_matrix_df and binary_matrix_ui.')

matrix_source = binary_matrix_df
if isinstance(matrix_source, pd.DataFrame):
    matrix = matrix_source.to_numpy(dtype=float)
    row_labels_full = list(matrix_source.index)
    col_labels_full = list(matrix_source.columns)
else:
    matrix = np.asarray(matrix_source, dtype=float)
    if matrix.ndim != 2:
        raise ValueError('binary_matrix_df must be a 2D table or array.')
    row_labels_full = list(range(matrix.shape[0]))
    col_labels_full = list(range(matrix.shape[1]))

kernel_candidate = binary_matrix_ui
if isinstance(kernel_candidate, pd.DataFrame):
    kernel_array = kernel_candidate.to_numpy(dtype=float)
else:
    kernel_array = np.asarray(kernel_candidate, dtype=float)

if kernel_array.ndim != 2:
    raise ValueError('binary_matrix_ui must be a 2D structure to act as a kernel.')

if kernel_array.size == 0:
    raise ValueError('binary_matrix_ui is empty; draw a pattern before running the analysis cell.')

selected_metrics = [m for m in METRICS_TO_RUN if m in AVAILABLE_METRICS]
if not selected_metrics:
    raise ValueError('METRICS_TO_RUN does not include any supported metrics.')

print(f'Source binary_matrix_df shape: {matrix.shape}')
print(f'Base kernel shape: {kernel_array.shape}')
print(f'Stride (Y, X): ({STRIDE_Y}, {STRIDE_X})')
print('Metrics to run:', ', '.join(AVAILABLE_METRICS[m]['label'] for m in selected_metrics))

def resize_kernel(arr, scale_y, scale_x):
    arr = np.asarray(arr, dtype=float)
    src_rows, src_cols = arr.shape
    tgt_rows = max(1, int(round(src_rows * scale_y)))
    tgt_cols = max(1, int(round(src_cols * scale_x)))
    if tgt_rows == src_rows and tgt_cols == src_cols:
        return arr.copy()
    y_grid = np.linspace(0, src_rows - 1, tgt_rows)
    x_grid = np.linspace(0, src_cols - 1, tgt_cols)
    y0 = np.floor(y_grid).astype(int)
    y1 = np.clip(y0 + 1, 0, src_rows - 1)
    wy = y_grid - y0
    x0 = np.floor(x_grid).astype(int)
    x1 = np.clip(x0 + 1, 0, src_cols - 1)
    wx = x_grid - x0
    top = (1 - wx) * arr[y0[:, None], x0] + wx * arr[y0[:, None], x1]
    bottom = (1 - wx) * arr[y1[:, None], x0] + wx * arr[y1[:, None], x1]
    return (1 - wy)[:, None] * top + wy[:, None] * bottom

def summarize_best(name, data, labels_y, labels_x):
    finite_mask = np.isfinite(data)
    if not finite_mask.any():
        print(f'{name}: no finite values to summarise.')
        return
    flat_index = np.nanargmax(data)
    r, c = divmod(flat_index, data.shape[1])
    print(f'{name} best: {data[r, c]:.3f} at top-left (row={labels_y[r]}, col={labels_x[c]})')

def top_matches(name, data, labels_y, labels_x, top_n, threshold=None):
    if top_n is None or top_n <= 0:
        return
    flat = data.reshape(-1)
    mask = np.isfinite(flat)
    if threshold is not None:
        mask &= flat >= threshold
    indices = np.nonzero(mask)[0]
    if indices.size == 0:
        print(f'{name}: no entries meet the threshold.')
        return
    ranked = indices[np.argsort(flat[indices])[::-1]]
    limit = min(top_n, ranked.size)
    print(f'{name} top {limit} positions:')
    for idx in ranked[:limit]:
        val = flat[idx]
        r, c = divmod(idx, data.shape[1])
        print(f'  score={val:.3f} at row={labels_y[r]}, col={labels_x[c]}')

def apply_threshold(data, threshold):
    if threshold is None:
        return data
    return np.where(data >= threshold, data, np.nan)

def finite_bounds(data):
    finite = data[np.isfinite(data)]
    if finite.size == 0:
        return 0.0, 1.0
    low = float(finite.min())
    high = float(finite.max())
    if low == high:
        high = low + 1e-9
    return low, high

def plot_matplotlib_single(title, data, threshold=None):
    fig, ax = plt.subplots(figsize=(9, 6))
    masked = apply_threshold(data, threshold)
    im = ax.imshow(masked, cmap=MPL_CMAP, origin='upper', aspect='auto')
    ax.set_title(title)
    ax.set_xlabel('time offset (columns)')
    ax.set_ylabel('pitch offset (rows)')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()

def plot_bokeh_single(title, data, threshold=None):
    masked = apply_threshold(data, threshold)
    low, high = finite_bounds(masked)
    fig = figure(
        title=title,
        x_range=(0, masked.shape[1]),
        y_range=(0, masked.shape[0]),
        width=900,
        height=600,
        tools='pan,wheel_zoom,reset,save',
    )
    mapper = LinearColorMapper(palette=Viridis256, low=low, high=high)
    fig.image(image=[np.flipud(masked)], x=0, y=0, dw=masked.shape[1], dh=masked.shape[0], color_mapper=mapper)
    fig.add_layout(ColorBar(color_mapper=mapper), 'right')
    fig.xaxis.axis_label = 'time offset (columns)'
    fig.yaxis.axis_label = 'pitch offset (rows, top=high)'
    show(fig)

def plot_kernel(title, kernel):
    if backend_to_use in ('plt', 'both'):
        fig, ax = plt.subplots(figsize=(9, 6))
        im = ax.imshow(kernel, cmap=MPL_CMAP, origin='upper', aspect='auto')
        ax.set_title(title)
        ax.set_xlabel('kernel columns')
        ax.set_ylabel('kernel rows')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.show()
    if backend_to_use in ('bokeh', 'both'):
        low, high = finite_bounds(kernel)
        fig = figure(
            title=title,
            x_range=(0, kernel.shape[1]),
            y_range=(0, kernel.shape[0]),
            width=900,
            height=600,
            tools='pan,wheel_zoom,reset,save',
        )
        mapper = LinearColorMapper(palette=Viridis256, low=low, high=high)
        fig.image(image=[np.flipud(kernel)], x=0, y=0, dw=kernel.shape[1], dh=kernel.shape[0], color_mapper=mapper)
        fig.add_layout(ColorBar(color_mapper=mapper), 'right')
        fig.xaxis.axis_label = 'kernel columns'
        fig.yaxis.axis_label = 'kernel rows (top=high)'
        show(fig)

scale_variants = []
seen_scales = set()
for axis in KERNEL_SCALE_AXES:
    axis_norm = axis.lower()
    if axis_norm not in {'x', 'y', 'both'}:
        raise ValueError(f'Unsupported axis {axis!r} in KERNEL_SCALE_AXES; use "x", "y", or "both".')
    for factor in KERNEL_SCALE_FACTORS:
        factor = float(factor)
        if factor <= 0:
            raise ValueError(f'Scale factor must be positive; received {factor}.')
        scale_y = factor if axis_norm in {'y', 'both'} else 1.0
        scale_x = factor if axis_norm in {'x', 'both'} else 1.0
        key = (round(scale_y, 6), round(scale_x, 6))
        if key in seen_scales:
            continue
        seen_scales.add(key)
        label = f'axis={axis_norm}, factor={factor:g}, scale_y={scale_y:.3f}, scale_x={scale_x:.3f}'
        scale_variants.append({'axis': axis_norm, 'factor': factor, 'scale_y': scale_y, 'scale_x': scale_x, 'label': label})

if (1.0, 1.0) not in seen_scales:
    scale_variants.insert(0, {
        'axis': 'both',
        'factor': 1.0,
        'scale_y': 1.0,
        'scale_x': 1.0,
        'label': 'axis=both, factor=1, scale_y=1.000, scale_x=1.000',
    })

print(f'Kernel scale variants to evaluate: {len(scale_variants)}')

all_results = {}
scaled_kernels = {}
last_variant_key = None
last_conv_raw_df = None

for variant in tqdm(scale_variants, desc='Kernel variants'):
    scale_y = variant['scale_y']
    scale_x = variant['scale_x']
    variant_key = variant['label']
    scaled_kernel = resize_kernel(kernel_array, scale_y, scale_x)
    tqdm.write(f'Processing {variant_key} -> shape={scaled_kernel.shape}')
    if PLOT_SCALED_KERNELS:
        plot_kernel(f'Scaled kernel ({variant_key})', scaled_kernel)

    if scaled_kernel.shape[0] > matrix.shape[0] or scaled_kernel.shape[1] > matrix.shape[1]:
        tqdm.write('  Skipped: scaled kernel is larger than the source matrix.')
        continue

    window_shape = scaled_kernel.shape
    out_rows = matrix.shape[0] - window_shape[0] + 1
    out_cols = matrix.shape[1] - window_shape[1] + 1
    if out_rows <= 0 or out_cols <= 0:
        tqdm.write('  Skipped: kernel cannot slide within the source matrix.')
        continue

    windows = np.lib.stride_tricks.sliding_window_view(matrix, window_shape)
    conv_scores_full = (windows * scaled_kernel).sum(axis=(-2, -1))

    metric_arrays = {}
    metric_dfs = {}

    row_positions = list(range(0, out_rows, max(1, int(STRIDE_Y))))
    col_positions = list(range(0, out_cols, max(1, int(STRIDE_X))))
    row_labels = [row_labels_full[pos] for pos in row_positions]
    col_labels = [col_labels_full[pos] for pos in col_positions]

    if 'normalized_overlap' in selected_metrics:
        kernel_weight = scaled_kernel.sum()
        if kernel_weight != 0:
            conv_norm_full = conv_scores_full / kernel_weight
        else:
            conv_norm_full = conv_scores_full.astype(float)
        conv_norm = conv_norm_full[np.ix_(row_positions, col_positions)]
        metric_arrays['normalized_overlap'] = conv_norm
        metric_dfs['normalized_overlap'] = pd.DataFrame(conv_norm, index=row_labels, columns=col_labels)
        conv_raw = conv_scores_full[np.ix_(row_positions, col_positions)]
        last_conv_raw_df = pd.DataFrame(conv_raw, index=row_labels, columns=col_labels)

    needs_cross = any(m in selected_metrics for m in ('cross_covariance', 'normalized_cross_correlation'))
    if needs_cross:
        windows_flat = windows.reshape(out_rows, out_cols, -1)
        kernel_flat = scaled_kernel.reshape(-1)
        kernel_mean = kernel_flat.mean()
        kernel_zero_mean = kernel_flat - kernel_mean
        kernel_norm_val = np.linalg.norm(kernel_zero_mean)
        window_means = windows_flat.mean(axis=2, keepdims=True)
        windows_zero_mean = windows_flat - window_means
        window_norms = np.linalg.norm(windows_zero_mean, axis=2)
        cross_cov_full = np.tensordot(windows_zero_mean, kernel_zero_mean, axes=([2], [0]))
        if 'cross_covariance' in selected_metrics:
            cross_cov = cross_cov_full[np.ix_(row_positions, col_positions)]
            metric_arrays['cross_covariance'] = cross_cov
            metric_dfs['cross_covariance'] = pd.DataFrame(cross_cov, index=row_labels, columns=col_labels)
        if 'normalized_cross_correlation' in selected_metrics:
            denominator = window_norms * kernel_norm_val
            norm_cross_full = np.divide(
                cross_cov_full,
                denominator,
                out=np.zeros_like(cross_cov_full),
                where=denominator > 0,
            )
            norm_cross = norm_cross_full[np.ix_(row_positions, col_positions)]
            metric_arrays['normalized_cross_correlation'] = norm_cross
            metric_dfs['normalized_cross_correlation'] = pd.DataFrame(norm_cross, index=row_labels, columns=col_labels)

    for metric_key in selected_metrics:
        metric_label = AVAILABLE_METRICS[metric_key]['label']
        data = metric_arrays.get(metric_key)
        if data is None:
            continue
        summarize_best(metric_label, data, row_labels, col_labels)
        threshold = PLOT_THRESHOLDS.get(metric_key)
        top_matches(metric_label, data, row_labels, col_labels, TOP_N_MATCHES, threshold)
        if backend_to_use == 'plt':
            plot_matplotlib_single(metric_label, data, threshold=threshold)
        elif backend_to_use == 'bokeh':
            plot_bokeh_single(metric_label, data, threshold=threshold)
        elif backend_to_use == 'both':
            plot_matplotlib_single(metric_label, data, threshold=threshold)
            plot_bokeh_single(metric_label, data, threshold=threshold)
        elif backend_to_use == 'none':
            pass
        else:
            msg = "Unknown backend {backend!r}. Available options: 'plt', 'bokeh', 'both', 'none'. Defaulting to Matplotlib."
            print(msg.format(backend=backend_to_use))
            plot_matplotlib_single(metric_label, data, threshold=threshold)

    all_results[variant_key] = metric_dfs
    scaled_kernels[variant_key] = scaled_kernel
    last_variant_key = variant_key

pattern_search_results = all_results
pattern_search_kernels = scaled_kernels
last_pattern_search_variant = last_variant_key

if last_variant_key is not None:
    last_metrics = all_results[last_variant_key]
    conv_norm_df = last_metrics.get('normalized_overlap')
    cross_cov_df = last_metrics.get('cross_covariance')
    norm_cross_df = last_metrics.get('normalized_cross_correlation')
else:
    conv_norm_df = None
    cross_cov_df = None
    norm_cross_df = None

conv_raw_df = last_conv_raw_df
print('Stored results in pattern_search_results; last variant key:', last_variant_key)


Source binary_matrix_df shape: (128, 920)
Base kernel shape: (4, 10)
Stride (Y, X): (1, 1)
Metrics to run: Normalised overlap
Kernel scale variants to evaluate: 5


Kernel variants:   0%|          | 0/5 [00:00<?, ?it/s]

Processing axis=x, factor=0.5, scale_y=1.000, scale_x=0.500 -> shape=(4, 5)
Normalised overlap best: 0.800 at top-left (row=49, col=380)
Normalised overlap top 20 positions:
  score=0.800 at row=78, col=230
  score=0.800 at row=61, col=106
  score=0.800 at row=50, col=536
  score=0.800 at row=52, col=360
  score=0.800 at row=52, col=616
  score=0.800 at row=55, col=842
  score=0.800 at row=56, col=122
  score=0.800 at row=56, col=144
  score=0.800 at row=56, col=210
  score=0.800 at row=57, col=568
  score=0.800 at row=57, col=824
  score=0.800 at row=59, col=906
  score=0.800 at row=61, col=206
  score=0.800 at row=76, col=774
  score=0.800 at row=61, col=254
  score=0.800 at row=61, col=582
  score=0.800 at row=61, col=782
  score=0.800 at row=61, col=810
  score=0.800 at row=63, col=506
  score=0.800 at row=64, col=290


Processing axis=x, factor=0.75, scale_y=1.000, scale_x=0.750 -> shape=(4, 8)
Normalised overlap best: 0.768 at top-left (row=55, col=841)
Normalised overlap top 18 positions:
  score=0.768 at row=78, col=229
  score=0.768 at row=55, col=841
  score=0.768 at row=64, col=797
  score=0.768 at row=64, col=289
  score=0.768 at row=61, col=809
  score=0.768 at row=61, col=781
  score=0.768 at row=61, col=253
  score=0.768 at row=59, col=905
  score=0.768 at row=76, col=773
  score=0.750 at row=57, col=566
  score=0.750 at row=57, col=822
  score=0.750 at row=50, col=534
  score=0.750 at row=52, col=349
  score=0.714 at row=56, col=685
  score=0.714 at row=68, col=413
  score=0.714 at row=54, col=549
  score=0.714 at row=54, col=309
  score=0.714 at row=52, col=377


Processing axis=x, factor=1, scale_y=1.000, scale_x=1.000 -> shape=(4, 10)
Normalised overlap best: 0.800 at top-left (row=50, col=532)
Normalised overlap top 20 positions:
  score=0.800 at row=59, col=904
  score=0.800 at row=64, col=288
  score=0.800 at row=52, col=348
  score=0.800 at row=55, col=840
  score=0.800 at row=56, col=560
  score=0.800 at row=57, col=564
  score=0.800 at row=57, col=820
  score=0.800 at row=57, col=836
  score=0.800 at row=78, col=228
  score=0.800 at row=61, col=252
  score=0.800 at row=61, col=780
  score=0.800 at row=61, col=808
  score=0.800 at row=50, col=532
  score=0.800 at row=64, col=796
  score=0.800 at row=76, col=772
  score=0.800 at row=66, col=272
  score=0.700 at row=54, col=308
  score=0.700 at row=56, col=684
  score=0.700 at row=71, col=686
  score=0.700 at row=56, col=559


Processing axis=x, factor=1.5, scale_y=1.000, scale_x=1.500 -> shape=(4, 15)
Normalised overlap best: 0.848 at top-left (row=55, col=552)
Normalised overlap top 20 positions:
  score=0.848 at row=55, col=552
  score=0.829 at row=56, col=556
  score=0.810 at row=55, col=551
  score=0.795 at row=61, col=805
  score=0.786 at row=56, col=557
  score=0.786 at row=61, col=204
  score=0.767 at row=66, col=268
  score=0.762 at row=56, col=140
  score=0.757 at row=61, col=756
  score=0.757 at row=71, col=508
  score=0.748 at row=61, col=806
  score=0.743 at row=61, col=804
  score=0.729 at row=56, col=141
  score=0.729 at row=59, col=901
  score=0.729 at row=66, col=267
  score=0.724 at row=61, col=778
  score=0.719 at row=59, col=902
  score=0.714 at row=61, col=205
  score=0.714 at row=52, col=780
  score=0.710 at row=54, col=548


Processing axis=x, factor=2, scale_y=1.000, scale_x=2.000 -> shape=(4, 20)
Normalised overlap best: 0.753 at top-left (row=66, col=264)
Normalised overlap top 4 positions:
  score=0.753 at row=66, col=264
  score=0.737 at row=55, col=548
  score=0.711 at row=66, col=263
  score=0.700 at row=59, col=900


Stored results in pattern_search_results; last variant key: axis=x, factor=2, scale_y=1.000, scale_x=2.000


### Pattern Search Metrics Overview

- **Normalised overlap** divides the raw overlap count by the sum of kernel values. With binary kernels this ranges from 0 (no overlap) to 1 (perfect alignment). Values above 1 only appear if the kernel contains weights greater than 1.
- **Cross-covariance** subtracts the local mean before scoring. The scale depends on the data magnitude; higher positive numbers indicate stronger alignment with the kernel's on/off pattern, while negative values suggest an inverted match.
- **Normalised cross-correlation** rescales cross-covariance by both vector norms, yielding scores in [-1, 1]. A value of 1 is a perfect positive match, 0 indicates no linear relationship, and -1 is a perfect inversion.
- <code>METRICS_TO_RUN</code> lets you choose which metrics to evaluate and plot. Remove entries to skip heavy calculations.
- <code>PLOT_THRESHOLDS</code> supplies a per-metric cutoff used for both the heatmaps and the top-N match listings.
- <code>KERNEL_SCALE_FACTORS</code> and <code>KERNEL_SCALE_AXES</code> generate scaled versions of the UI kernel so you can test augmentations/diminutions. Enable <code>PLOT_SCALED_KERNELS</code> to visualise each variant with the same backend setting.
- The <code>tqdm</code> progress bar tracks kernel variants; detailed results live in <code>pattern_search_results[variant_key][metric_key]</code>, and the last run variant remains accessible through <code>conv_norm_df</code>, <code>cross_cov_df</code>, <code>norm_cross_df</code>, and <code>conv_raw_df</code>.



to do:

Sliding-window Hamming distance or Sum of Absolute/ squared Differences computed on the binary matrices (optionally vectorized with sliding_window_view).

Self-similarity matrices or recurrence plots to locate repeated motifs independent of a fixed kernel size.

Dynamic Time Warping or sequence alignment on extracted pitch/time contours to tolerate stretching or small rhythmic deviations.

Hashing / locality-sensitive hashing on flattened windows for faster approximate lookups when scanning large corpora.